# 📘 DATABASE MANAGEMENT AND DESIGN
### *A Practical Learning Series*

# Notebook 3.2 — SQL and Data Analysis — Intermediate Level
### Functions, GROUP BY, HAVING

**By Amin Amirkhalili** — Business & Data Analyst

*Based on Chapter 6 (Section 6-5) of:* **Database Management and Design** — Gove Allen, PhD · Gary Hansen, PhD · Robert Jackson, PhD


## 🗺️ Series Roadmap

This notebook is part of a practical learning series on database management and design. The series follows a logical progression — from understanding business needs, through conceptual modelling, all the way to writing SQL code on real databases.

| | |
|---|---|
| **1 — Fundamentals** | |
| Notebook 1 | Introduction: The Database Is the Heart of Information Systems |
| **2 — Database Management** | |
| Notebook 2.1 | Conceptual Data Model: From Conceptual Design to a Relational Model |
| Notebook 2.2 | Database Design: From Schema Creation to Data Management |
| **3 — SQL & Data Analysis** | |
| Notebook 3.1 | Beginner Level: SELECT, FROM, WHERE, ORDER BY |
| **Notebook 3.2** | Intermediate Level: Functions, GROUP BY, HAVING **◀ You are here** |
| Notebook 3.3 | Advanced Level I: Joining Tables — Inner & Outer Joins |
| Notebook 3.4 | Advanced Level II: Subqueries, Views, Temp Tables, CTEs |
| Notebook 3.5 | Data Cleaning with SQL *(coming soon)* |
| **4 — Data Mining** | |
| Notebook 4.1 | SQL in Data Mining *(coming soon)* |

## 📌 What This Notebook Covers

Functions inside SELECT (arithmetic operations, arithmetic functions, aggregation functions, string functions), GROUP BY, PARTITION BY, and HAVING.

## 💻 How to Use This Notebook

This is the **live practice version** — every query below runs for real.

1. Run the setup cell below **first** (▶ button on the left). It builds a small sample database with the same tables used in the series.
2. Run each query cell as you read. Change the queries — experiment!
3. To start fresh, just run the setup cell again.

> **Note on SQL dialects:** the reading copy of this notebook uses **SQL Server (SSMS)**. Here we run **SQLite** (built into Colab, no installation needed). The two are almost identical; wherever the syntax differs (e.g. `TOP` vs `LIMIT`, `+` vs `||` for text), a note shows both versions.

In [1]:
#@title ▶️ Run this cell first — it builds the practice database { display-mode: "form" }
# This cell creates a small sample database (SQLite) with the same tables used
# in the notebook: Customer, Manufacturer, Product, Sale, SaleItem, Employee,
# SalaryEmployee, WageEmployee, Purchase, Amin, SimplifiedSales, and more.
# It also defines q("...") which runs a SQL query and shows the result.

import sqlite3, math, datetime
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE Customer (
  CustomerID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  StreetAddress VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Country VARCHAR(30), Phone VARCHAR(20),
  PRIMARY KEY (CustomerID));

CREATE TABLE Manufacturer (
  ManufacturerID INT NOT NULL, ManufacturerName VARCHAR(50),
  Address VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Phone VARCHAR(20),
  PRIMARY KEY (ManufacturerID));

CREATE TABLE Product (
  ProductID INT NOT NULL, ProductName VARCHAR(60), ManufacturerID INT,
  Category VARCHAR(30), Color VARCHAR(20), Gender VARCHAR(1),
  ListPrice NUMERIC(8,2), Description VARCHAR(120),
  PRIMARY KEY (ProductID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Sale (
  SaleID INT NOT NULL, SaleDate DATE NOT NULL,
  Tax NUMERIC(8,2), Shipping NUMERIC(8,2), CustomerID INT,
  PRIMARY KEY (SaleID),
  FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID));

CREATE TABLE SaleItem (
  SaleID INT NOT NULL, ProductID INT NOT NULL, ItemSize NUMERIC(3,1) NOT NULL,
  Quantity INT NOT NULL, SalePrice NUMERIC(8,2) NOT NULL,
  PRIMARY KEY (SaleID, ProductID, ItemSize),
  FOREIGN KEY (SaleID) REFERENCES Sale(SaleID),
  FOREIGN KEY (ProductID) REFERENCES Product(ProductID));

CREATE TABLE Employee (
  EmployeeID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, City VARCHAR(40), ManagerID INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (ManagerID) REFERENCES Employee(EmployeeID));

CREATE TABLE SalaryEmployee (
  EmployeeID INT NOT NULL, Salary NUMERIC(10,2),
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE WageEmployee (
  EmployeeID INT NOT NULL, Wage NUMERIC(8,2), MaxHours INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE Purchase (
  PurchaseID INT NOT NULL, PurchaseDate DATE, EmployeeID INT, ManufacturerID INT,
  PRIMARY KEY (PurchaseID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Amin (
  ID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, Gender VARCHAR(10), PRIMARY KEY (ID));

CREATE TABLE Population (
  Country VARCHAR(40), City VARCHAR(40), Population INT);

CREATE TABLE CovidDeaths (
  location VARCHAR(40), continent VARCHAR(20), date DATE,
  total_cases INT, population INT);

INSERT INTO Manufacturer VALUES
 (1,'Nike','1 Bowerman Dr','Boston','NJ','07101','555-0101'),
 (2,'Adidas','5 Sport Ave','Boston','NJ','07102','555-0102'),
 (3,'Puma','9 Cat St','Chicago','IL','60601','555-0103'),
 (4,'Fila','3 Retro Rd','San Diego','CA','92101','555-0104'),
 (5,'Fashion Lab','7 Style Blvd','Los Angeles','CA','90001','555-0105'),
 (6,'GAP','11 Cotton Way','Concord','NH','03301','555-0106');

INSERT INTO Customer VALUES
 (1,'Sara','Lopez','12 Main St','Boston','VA','23220','USA','555-1001'),
 (2,'Amin','Amirkhalili','34 Oak Ave','Boston','MA','02108','USA','555-1002'),
 (3,'Kim','Nguyen','56 Pine Rd','Chicago','IL','60614','USA','555-1003'),
 (4,'John','Smith','78 Elm St','Richmond','VA','23230','USA','555-1004'),
 (5,'George','Brown','90 Cedar Ln','Springfield','IL','62701','USA','555-1005'),
 (6,'Kevin','Miller','21 Birch Ct','San Diego','CA','92103','USA','555-1006'),
 (7,'Sam','Wilson','43 Maple Dr','Portland','OR','97201','USA','555-1007'),
 (8,'Elli','Rahimi','65 Walnut St','Los Angeles','CA','90012','USA','555-1008');

INSERT INTO Product VALUES
 (1,'AirRun Sneaker',1,'Sneakers','Black','M',120,'Running sneaker'),
 (2,'AirRun Sneaker W',1,'Sneakers','White','F',115,'Running sneaker'),
 (3,'ClassicSandal',1,'Sandals','Brown','F',60,'Summer sandal'),
 (4,'StreetBoot',2,'Boots','Black','M',150,'Urban boot'),
 (5,'TrailBoot',2,'Boots','Brown','F',160,'Hiking boot'),
 (6,'SpeedCat Sneaker',3,'Sneakers','Red','M',95,'Classic sneaker'),
 (7,'BeachSandal',3,'Sandals','Gold','F',45,'Beach sandal'),
 (8,'RetroRunner',4,'Sneakers','Black','F',85,'Retro sneaker'),
 (9,'HeritageBoot',4,'Boots','Tan','M',140,'Heritage boot'),
 (10,'GlamHeel',5,'Heels','Black','F',130,'Evening heel'),
 (11,'CityFlat',5,'Flats','Blue','F',70,'Comfort flat'),
 (12,'CanvasClassic',6,'Sneakers','White','M',55,'Canvas sneaker'),
 (13,'WinterBoot',6,'Boots','Black','F',110,'Insulated boot');

INSERT INTO Sale VALUES
 (1001,'2026-07-03',8.40,5.00,1),
 (1002,'2026-07-02',6.20,5.00,2),
 (1003,'2026-06-28',12.10,0.00,3),
 (1004,'2026-06-15',4.75,5.00,4),
 (1005,'2026-05-30',9.90,7.50,2),
 (1006,'2026-01-18',7.25,5.00,5),
 (1007,'2026-01-05',3.60,0.00,6),
 (1008,'2025-12-20',10.10,5.00,1),
 (1009,'2025-12-08',5.45,5.00,8),
 (1010,'2026-07-10',6.80,0.00,3);

INSERT INTO SaleItem VALUES
 (1001,1,10.0,10,100.00),(1001,4,9.5,5,200.00),
 (1002,6,8.0,1,150.00),
 (1003,2,7.5,2,110.00),(1003,7,6.0,1,42.00),
 (1004,8,7.0,1,80.00),
 (1005,10,6.5,1,125.00),(1005,11,6.0,2,65.00),
 (1006,12,11.0,3,50.00),(1006,1,10.5,1,110.00),
 (1007,9,10.5,1,135.00),
 (1008,13,8.5,2,105.00),(1008,3,7.0,1,55.00),
 (1009,5,8.0,1,155.00),
 (1010,1,9.0,1,118.00);

INSERT INTO Employee VALUES
 (1,'Alan','Stone',55,'Boston',NULL),
 (2,'Ahmad','Karimi',41,'Boston',1),
 (3,'Elli','Moore',38,'Chicago',1),
 (4,'Luke','Perry',29,'San Diego',2),
 (5,'James','Chen',33,'San Diego',2),
 (6,'Alex','Diaz',26,'Portland',3);

INSERT INTO SalaryEmployee VALUES
 (1,95000),(2,72000),(3,68000),(4,80000);

INSERT INTO WageEmployee VALUES
 (5,28.50,40),(6,22.00,30);

INSERT INTO Purchase VALUES
 (501,'2025-12-05',2,1),
 (502,'2025-12-12',4,2),
 (503,'2025-12-27',5,3),
 (504,'2026-02-10',2,4),
 (505,'2026-03-22',3,5),
 (506,'2025-12-30',NULL,6);

INSERT INTO Amin VALUES
 (1,'Amin','Amirkhalili',30,'Male'),
 (2,'Amin','Amirkhalili',50,'Male'),
 (3,'Sara','Lopez',27,'Female'),
 (4,'Elli','Rahimi',35,'Female'),
 (5,'Sadra','Kazemi',22,'Male'),
 (6,'Hossein','Nouri',44,'Male'),
 (7,'Kim','Nguyen',31,'Female');

INSERT INTO Population VALUES
 ('Canada','Toronto',2794356),('Canada','Montreal',1762949),
 ('Canada','Vancouver',662248),('USA','New York',8804190),
 ('USA','Los Angeles',3898747),('USA','Chicago',2746388),
 ('Germany','Berlin',3644826),('Germany','Munich',1471508);

INSERT INTO CovidDeaths VALUES
 ('Canada','North America','2021-06-01',1385000,38000000),
 ('Canada','North America','2021-06-15',1401000,38000000),
 ('Canada','North America','2021-07-01',1412000,38000000),
 ('Germany','Europe','2021-06-01',3700000,83000000),
 ('Germany','Europe','2021-06-15',3717000,83000000),
 ('Germany','Europe','2021-07-01',3730000,83000000),
 ('Iran','Asia','2021-06-01',2950000,85000000),
 ('Iran','Asia','2021-06-15',3020000,85000000),
 ('Iran','Asia','2021-07-01',3230000,85000000);

CREATE TABLE SimplifiedSales AS
SELECT s.SaleID, s.SaleDate, s.CustomerID, p.ProductName, p.Category, p.Color,
       m.ManufacturerName, si.Quantity, si.SalePrice
FROM Sale s
JOIN SaleItem si ON s.SaleID = si.SaleID
JOIN Product p  ON si.ProductID = p.ProductID
JOIN Manufacturer m ON p.ManufacturerID = m.ManufacturerID;
""")

# --- Register SQL-Server-style functions so the book's code runs in SQLite ---
conn.create_function("LEN", 1, lambda s: len(s) if s is not None else None)
conn.create_function("CHARINDEX", 2, lambda sub, s: (s.find(sub) + 1) if (s is not None and sub is not None) else None)
conn.create_function("SQRT", 1, lambda x: math.sqrt(x) if x is not None else None)
conn.create_function("POWER", 2, lambda x, n: x ** n if x is not None else None)
conn.create_function("CEILING", 1, lambda x: math.ceil(x) if x is not None else None)
conn.create_function("FLOOR", 1, lambda x: math.floor(x) if x is not None else None)
conn.create_function("GETDATE", 0, lambda: datetime.date.today().isoformat())
conn.create_function("MONTH", 1, lambda d: int(str(d)[5:7]) if d else None)
conn.create_function("YEAR", 1, lambda d: int(str(d)[0:4]) if d else None)

class _StringAgg:
    def __init__(self): self.items = []
    def step(self, value, sep): self.sep = sep; self.items.append(str(value))
    def finalize(self): return self.sep.join(self.items) if self.items else None
conn.create_aggregate("STRING_AGG", 2, _StringAgg)

def q(sql):
    """Run a SQL query and show the result as a table."""
    sql_stripped = sql.strip().rstrip(";").strip()
    first = sql_stripped.split()[0].upper() if sql_stripped else ""
    if first in ("SELECT", "WITH"):
        return pd.read_sql_query(sql, conn)
    conn.executescript(sql)
    print("OK —", first, "executed.")

print("✅ Practice database is ready. Use q(\"...\") to run SQL. Example:")
q("SELECT * FROM Customer LIMIT 3")


✅ Practice database is ready. Use q("...") to run SQL. Example:


,CustomerID,FirstName,LastName,StreetAddress,City,State,PostalCode,Country,Phone
0,1,Sara,Lopez,12 Main St,Boston,VA,23220,USA,555-1001
1,2,Amin,Amirkhalili,34 Oak Ave,Boston,MA,02108,USA,555-1002
2,3,Kim,Nguyen,56 Pine Rd,Chicago,IL,60614,USA,555-1003


## 1. Functions [in SELECT]

There are four groups of functions we use in SELECT:

1. Arithmetic operations
2. Arithmetic functions
3. Aggregation functions
4. String functions

### 1.1 Arithmetic operations: `+  −  ÷  ×  %`

In [2]:
q("""
SELECT SaleID, (Shipping + Tax) AS AddonCost FROM Sale
""")

,SaleID,AddonCost
0,1001,13.40
1,1002,11.20
2,1003,12.10
3,1004,9.75
4,1005,17.40
5,1006,12.25
6,1007,3.60
7,1008,15.10
8,1009,10.45
9,1010,6.80


**`+` in strings** — in SQL Server: `SELECT LastName + ', ' + FirstName AS CustomerName FROM Customer`.
SQLite uses `||` for text concatenation:

In [3]:
q("""
SELECT LastName || ', ' || FirstName AS CustomerName FROM Customer
""")

,CustomerName
0,"Lopez, Sara"
1,"Amirkhalili, Amin"
2,"Nguyen, Kim"
3,"Smith, John"
4,"Brown, George"
5,"Miller, Kevin"
6,"Wilson, Sam"
7,"Rahimi, Elli"


### 1.2 Arithmetic functions

```
ABS(x)      : ABS(-10)         = 10
SQRT(x)     : SQRT(16)         = 4
ROUND(x, n) : ROUND(4.032, 2)  = 4.03
CEILING(x)  : CEILING(4.05)    = 5
FLOOR(x)    : FLOOR(4.09)      = 4
POWER(x, n) : POWER(2, 2)      = 4
```

In [4]:
q("""
SELECT ABS(-10), SQRT(16), ROUND(4.032, 2), CEILING(4.05), FLOOR(4.09), POWER(2, 2)
""")

,ABS(-10),SQRT(16),"ROUND(4.032, 2)",CEILING(4.05),FLOOR(4.09),"POWER(2, 2)"
0,10,4.0,4.03,5,4,4


In [5]:
q("""
SELECT POWER(Age, 2) FROM Amin
""")

,"POWER(Age, 2)"
0,900
1,2500
2,729
3,1225
4,484
5,1936
6,961


### 1.3 Aggregation functions

```
MIN(x)  MAX(x)  SUM(x)  AVG(x)  COUNT(x)
```

They usually come alone, or together with other aggregation functions, in SELECT. **BUT**, if they come with other attributes, we must use GROUP BY. [They can also come in the HAVING clause.] (See Section 2.)

In [6]:
q("""
SELECT AVG(Age) FROM Amin
""")

,AVG(Age)
0,34.142857


In [7]:
q("""
SELECT AVG(Age), Gender FROM Amin
GROUP BY Gender
""")

,AVG(Age),Gender
0,31.0,Female
1,36.5,Male


### 1.4 String functions

| Function | What it does | Example |
|---|---|---|
| `LEN(x)` | length of the string *(SQL Server; SQLite: `LENGTH`)* | `LEN('My house is white.')` → 18 |
| `UPPER(x)` | all characters upper case | `UPPER('George')` → GEORGE |
| `LOWER(x)` | all characters lower case | `LOWER('McDougal')` → mcdougal |
| `CHARINDEX(v, x)` | first location of v in x *(SQLite: `INSTR`)* | `CHARINDEX('hi', 'My white house')` → 5 |
| `LTRIM(x)` | removes leading spaces | `LTRIM('   Bill')` → Bill |
| `RTRIM(x)` | removes trailing spaces | `RTRIM('Shoes   ')` → Shoes |
| `TRIM(x)` | removes spaces from both sides | `TRIM('  Shoes  ')` → Shoes |
| `CONCAT(x, y, ...)` | concatenates strings | `CONCAT(LastName, ', ', FirstName)` → Jones, Bill |
| `SUBSTRING(x, n, m)` | m characters from the nth *(SQLite: `SUBSTR`)* | `SUBSTRING('My White House', 4, 5)` → White |
| `LEFT(x, n)` | first n characters | `LEFT('Bill Jones', 2)` → Bi |
| `RIGHT(x, n)` | last n characters | `RIGHT('Bill Jones', 2)` → es |
| `REPLACE(x, 'A', 'B')` | replaces every A with B | `REPLACE('shoe store', 'shoe', 'shirt')` → shirt store |

*(In this notebook, the SQL-Server names LEN and CHARINDEX are registered for you, so both spellings work. LEFT/RIGHT are reserved words in SQLite — use `SUBSTR(x, 1, n)` and `SUBSTR(x, -n)` instead.)*

In [8]:
q("""
SELECT LEN('My house is white.') AS L,
       UPPER('George') AS U,
       CHARINDEX('hi', 'My white house') AS C,
       SUBSTR('My White House', 4, 5) AS S,
       SUBSTR('Bill Jones', 1, 2) AS Lft,   -- LEFT('Bill Jones', 2)
       SUBSTR('Bill Jones', -2) AS Rgt,     -- RIGHT('Bill Jones', 2)
       REPLACE('Good shoes at the shoe store', 'shoe', 'shirt') AS Rep
""")

,L,U,C,S,Lft,Rgt,Rep
0,18,GEORGE,5,White,Bi,es,Good shirts at the shirt store


## 2. GROUP BY

As mentioned before, GROUP BY is usually used with aggregation functions such as COUNT(), SUM(), MIN(), MAX(), and AVG().

**Query: what is the average list price of products per category?**

In [9]:
q("""
SELECT Category, AVG(ListPrice) FROM Product
GROUP BY Category
""")

,Category,AVG(ListPrice)
0,Boots,140.0
1,Flats,70.0
2,Heels,130.0
3,Sandals,52.5
4,Sneakers,94.0


### When is it good to use GROUP BY?

**a) (Basic) Sometimes we don't need aggregation but still want to group by something.**

In [10]:
q("""
SELECT Gender FROM Amin
GROUP BY Gender
""")

,Gender
0,Female
1,Male


**b) (Pro) Where there is various data for one single item.** For example, one student has various grades. Or one customer has different orders. Then we might ask:

- What was the maximum sale for each customer so far?

In [11]:
q("""
SELECT MAX(SalePrice), CustomerID FROM SimplifiedSales
GROUP BY CustomerID
""")

,MAX(SalePrice),CustomerID
0,200,1
1,150,2
2,118,3
3,80,4
4,110,5
5,135,6
6,155,8


### It's also good to know when we do **not** need GROUP BY

**a) When we are not looking for any aggregation information:**

In [12]:
q("""
SELECT FirstName, LastName
FROM Customer
""")

,FirstName,LastName
0,Sara,Lopez
1,Amin,Amirkhalili
2,Kim,Nguyen
3,John,Smith
4,George,Brown
5,Kevin,Miller
6,Sam,Wilson
7,Elli,Rahimi


**b) When we are looking for a single aggregation for the whole table:**

In [13]:
q("""
SELECT SUM(SalePrice) FROM SaleItem
""")

,SUM(SalePrice)
0,1600


**c) When we are looking for a single aggregation for only one row:**

In [14]:
q("""
SELECT CustomerID, SUM(SalePrice)
FROM SimplifiedSales
WHERE CustomerID = 1
""")

,CustomerID,SUM(SalePrice)
0,1,460


> ### 💡 Golden rule of GROUP BY
> Conditions: (1) we want to use an aggregation function, and (2) we want to use GROUP BY.
> **If we want to use any attribute in SELECT, we must have that attribute in the GROUP BY clause (or it must also be aggregated).**

Look at this example — it is wrong (SQL Server rejects it; SQLite silently picks an arbitrary row, which is just as dangerous):

```sql
SELECT Category, ProductName, AVG(ListPrice)
FROM Product
GROUP BY Category;    -- ✗ ProductName is not in GROUP BY
```

Correction:

In [15]:
q("""
SELECT Category, ProductName, AVG(ListPrice)
FROM Product
GROUP BY Category, ProductName
""")

,Category,ProductName,AVG(ListPrice)
0,Boots,HeritageBoot,140.0
1,Boots,StreetBoot,150.0
2,Boots,TrailBoot,160.0
3,Boots,WinterBoot,110.0
4,Flats,CityFlat,70.0
5,Heels,GlamHeel,130.0
6,Sandals,BeachSandal,45.0
7,Sandals,ClassicSandal,60.0
8,Sneakers,AirRun Sneaker,120.0
9,Sneakers,AirRun Sneaker W,115.0


When you add more attributes to the GROUP BY clause, the results become more specific. When you have only one attribute, the grouping is more general:

In [16]:
q("""
SELECT Category, AVG(ListPrice)
FROM Product
GROUP BY Category
""")

,Category,AVG(ListPrice)
0,Boots,140.0
1,Flats,70.0
2,Heels,130.0
3,Sandals,52.5
4,Sneakers,94.0


### More examples

**Query: what groups of products exist?** [No aggregation required, but grouping intended — basic mode (a).]

In [17]:
q("""
SELECT Category
FROM Product
GROUP BY Category
""")

,Category
0,Boots
1,Flats
2,Heels
3,Sandals
4,Sneakers


**Query: for each product category, what is the highest list price for a product in that category?** [Aggregation required — pro mode (b).]

In [18]:
q("""
SELECT Category,
       MAX(ListPrice) AS MaxListPrice
FROM Product
GROUP BY Category
""")

,Category,MaxListPrice
0,Boots,160
1,Flats,70
2,Heels,130
3,Sandals,60
4,Sneakers,120


**Query: what is the number of people per gender?** (Pro mode.)

In [19]:
q("""
SELECT Gender, COUNT(Gender) FROM Amin
GROUP BY Gender
""")

,Gender,COUNT(Gender)
0,Female,3
1,Male,4


### Using ORDER BY with GROUP BY

**Query: what is the average price of products in each category? [sorted]**

In [20]:
q("""
SELECT Category, AVG(ListPrice) AS Avg_Price
FROM Product
GROUP BY Category
ORDER BY AVG(ListPrice) DESC
""")

,Category,Avg_Price
0,Boots,140.0
1,Heels,130.0
2,Sneakers,94.0
3,Flats,70.0
4,Sandals,52.5


### Using WHERE with GROUP BY

**Query: for each product category, what is the average list price of black shoes?**

In [21]:
q("""
SELECT Category, AVG(ListPrice) FROM Product
WHERE Color = 'Black'
GROUP BY Category
""")

,Category,AVG(ListPrice)
0,Boots,130.0
1,Heels,130.0
2,Sneakers,102.5


### PARTITION BY

With GROUP BY, the number of rows is reduced. But with PARTITION BY, you can have aggregation info for each row **while keeping all the rows**.

In [22]:
q("""
SELECT Gender, AVG(Age) AS AgeAvg FROM Amin
GROUP BY Gender
""")

,Gender,AgeAvg
0,Female,31.0
1,Male,36.5


Now, look at PARTITION BY:

In [23]:
q("""
SELECT FirstName, LastName, Age, Gender,
       AVG(Age) OVER (PARTITION BY Gender) AS AgeAvg
FROM Amin
""")

,FirstName,LastName,Age,Gender,AgeAvg
0,Sara,Lopez,27,Female,31.0
1,Elli,Rahimi,35,Female,31.0
2,Kim,Nguyen,31,Female,31.0
3,Amin,Amirkhalili,30,Male,36.5
4,Amin,Amirkhalili,50,Male,36.5
5,Sadra,Kazemi,22,Male,36.5
6,Hossein,Nouri,44,Male,36.5


Not only do we have the age average — we can still see all the rows.

PARTITION BY usually comes with these functions:

```
AVG()        OVER (PARTITION BY ...)
SUM()        OVER (PARTITION BY ...)
COUNT()      OVER (PARTITION BY ...)
ROW_NUMBER() OVER (PARTITION BY ...)   -- example in Notebook 3.4
RANK()       OVER (PARTITION BY ...)
```

## 3. HAVING

HAVING is a limitation/condition **after** we group something. (WHERE is a limitation/condition **before** we group.)

**Query: for each category containing more than N products, what is the maximum list price in that category?**

*(The textbook example uses N = 50; our sample database is small, so we use N = 2.)*

In [24]:
q("""
SELECT Category,
       MAX(ListPrice) AS MaxListPrice
FROM Product
GROUP BY Category
HAVING COUNT(*) > 2
""")

,Category,MaxListPrice
0,Boots,160
1,Sneakers,120


**The difference between the WHERE clause and the HAVING clause is that the WHERE clause is applied to *rows*, while the HAVING clause is applied to *groups*.**

### Using WHERE and HAVING with GROUP BY

**Query: for each product category, what is the average list price of black shoes? Consider only those categories having no product priced over $150.**

In [25]:
q("""
SELECT Category, AVG(ListPrice) FROM Product
WHERE Color = 'Black'
GROUP BY Category
HAVING MAX(ListPrice) <= 150
""")

,Category,AVG(ListPrice)
0,Boots,130.0
1,Heels,130.0
2,Sneakers,102.5


> ### 💡 Remember
> If you want to use any attribute in HAVING (such as price), it must be aggregated. So we write `MAX(ListPrice)`, not `ListPrice`. The reason: after grouping, there are no longer individual rows — they are groups now.
> In HAVING, the only things that are OK are: any attribute already used in the GROUP BY clause, and any attribute inside an aggregation function.

In [26]:
# the textbook uses < 30; our sample ages average a bit higher, so we use < 35
q("""
SELECT Gender, AVG(Age) FROM Amin
GROUP BY Gender
HAVING AVG(Age) < 35
""")

,Gender,AVG(Age)
0,Female,31.0


In [27]:
q("""
SELECT Gender, AVG(Age) FROM Amin
GROUP BY Gender
HAVING Gender LIKE 'Male'
""")

,Gender,AVG(Age)
0,Male,36.5


---
### 🎯 Your turn — practice ideas

1. What is the total sale value (`SalePrice * Quantity`) per manufacturer, highest first?
2. Which categories have an average list price above $100?
3. Show each person in `Amin` with the count of people of the same gender next to them (PARTITION BY).

---
⏭️ **Coming up in Notebook 3.3:** joining tables — inner and outer joins.